In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

from combine_features import read_data

In [29]:
df = read_data("../rhea-soil-nutrient-prediction-challenge/Train.csv")
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Cu,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,5.826,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,4.346,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.657,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.376,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.351,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN


In [30]:
encoder = LabelEncoder()
df['Depth'] = encoder.fit_transform(df['Depth_cm'])
df.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True)
df.head()

,Longitude,Latitude,ph,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,...,Fe,Mg,Mn,N,P,K,Na,S,Zn,Depth
0,37.65189,-3.15440,6.405,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,1477.0,...,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN,1
1,37.63612,-3.08585,6.419,14.202173,3.902236,-7.263155,15.240530,25.774622,105.626890,1513.5,...,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN,1
2,39.55580,-2.67218,8.388,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2198.0,...,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN,1
3,39.55477,-2.67196,8.302,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2258.0,...,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN,1
4,39.55477,-2.67196,8.292,14.202173,3.902236,-7.263155,21.969696,30.452652,63.580307,2258.0,...,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN,1


In [31]:
def get_S(row):
    return row['N'] / 12.5

def get_B(row):
    value = row['N'] / 1000
    if row['Ca'] > 1500:
        return value * 0.6
    return value

def get_Zn(row):
    value = row['Mn'] / 10
    if row['ph'] > 6.0:
        penalty = (row['ph'] - 6.0) * 0.5
        value *= (1 - min(penalty, 0.9))
    return value

def get_Na(row):
    return 0

def get_P(row):
    return 0

In [32]:
df['S'] = df.apply(get_S, axis=1)
df['B'] = df.apply(get_B, axis=1)
df['Zn'] = df.apply(get_Zn, axis=1)
df['Na'] = df.apply(get_Na, axis=1)
df['P'] = df.apply(get_P, axis=1)

In [33]:
df.dropna(axis=0, inplace=True)

In [34]:
target = [
        "Al",
        "B",
        "Ca",
        "Cu",
        "Fe",
        "Mg",
        "Mn",
        "N",
        "P",
        "K",
        "Na",
        "S",
        "Zn",
    ]
x = df.drop(columns=target, errors='ignore')
y_columns = df.columns.difference(x.columns)
y = df[y_columns]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [35]:
x_train.head()

,Longitude,Latitude,ph,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,B11,...,Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,C_organic,C_total,Depth
6489,42.73327,9.63641,8.360,12.434945,-0.612673,-4.526182,11.579545,26.409090,63.800938,2852.5,...,781.409091,2242.354545,2469.618182,40162.590909,27029.100000,9034.509091,675.527273,24.92,37.41,0
11881,48.70980,-18.27420,4.862,5.251518,-1.973700,-2.899782,15.196970,25.528410,127.234060,1452.0,...,0.000000,566.727273,593.090909,31748.763636,7640.372727,8651.509091,783.772727,42.07,42.65,0
8746,26.51750,-21.79080,7.003,56.498218,2.144745,7.334409,13.214015,29.979166,32.133892,2864.0,...,0.000000,473.781818,0.000000,0.000000,11428.600000,44759.450000,0.000000,22.35,26.94,1
8775,26.52350,-21.77120,8.409,56.498218,2.144745,7.334409,13.214015,29.979166,32.133892,3224.0,...,0.000000,473.781818,0.000000,0.000000,11428.600000,44759.450000,0.000000,14.50,22.31,1
11156,35.28978,-22.85395,6.318,9.193700,-1.378309,-7.520327,17.833334,30.005682,70.289390,3279.0,...,623.954545,503.063636,1355.866667,69430.263636,9185.563636,18506.954545,1157.872727,6.25,6.63,0


In [36]:
y_train.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
6489,308.492,0.000822,13740.489,4.572,35.494,530.812,1033.363,127.708,1.37,0,0,0.1096,1.277080
11881,1142.784,0.003250,323.494,2.271,192.366,67.531,96.545,102.805,3.25,0,0,0.2600,10.280500
8746,684.280,0.001158,3938.085,5.797,105.015,1029.506,1604.786,171.618,1.93,0,0,0.1544,8.555157
8775,298.667,0.001020,11769.776,5.077,117.038,896.004,722.193,71.026,1.70,0,0,0.1360,0.710260
11156,510.097,0.000500,638.328,0.566,62.306,47.994,88.265,101.614,0.50,0,0,0.0400,8.545737


In [44]:
models = {}
for col in target:
    model = RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=10)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    models[col] = model
    print(f"{col}: RMSE = {rmse}")

Al: RMSE = 110.14797164686746
B: RMSE = 0.0001892368735292506
Ca: RMSE = 663.5741713107005
Cu: RMSE = 0.6389526208977502
Fe: RMSE = 24.438591559970263
Mg: RMSE = 128.63621425758566
Mn: RMSE = 35.773890324301966
N: RMSE = 0.15907339094470327
P: RMSE = 0.0
K: RMSE = 89.80198830247853
Na: RMSE = 0.0
S: RMSE = 0.012727994304984488
Zn: RMSE = 2.8391401362254354


In [40]:
log_transform_cols = ['Ca', 'Mg', 'Cu', 'K', 'Zn']

In [41]:
log_transform_models = {}
for col in log_transform_cols:
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    model = TransformedTargetRegressor(regressor=rf, func=np.log1p, inverse_func=np.expm1)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    log_transform_models[col] = model
    print(f"{col}: RMSE = {rmse}")

Ca: RMSE = 646.234551601985
Mg: RMSE = 123.682192371565
Cu: RMSE = 0.6278571865945382
K: RMSE = 83.42459358543753
Zn: RMSE = 2.8049308246466818


Ca Model

In [49]:
ca = models['Ca']
fi = pd.DataFrame(ca.feature_importances_, index=x_train.columns, columns=['Feature Importance'])
fi.sort_values(by='Feature Importance', ascending=False, inplace=True)
print(f"Feature importance for Ca:")
print(fi.head(10))

Feature importance for Ca:
                               Feature Importance
ph                                       0.703195
C_total                                  0.146023
Longitude                                0.038856
Other fruits, n.e.c.                     0.034474
C_organic                                0.028523
Groundnuts, excluding shelled            0.016229
Latitude                                 0.012123
elevation                                0.005216
tmin_avg                                 0.002569
B8                                       0.001791
